In [1]:
%load_ext autoreload
%autoreload 2

In [2]:

from pyspark_data_provenance import (
    data_provenance_enabled, 
    build_data_provenance_session
)

In [3]:
from pyspark.sql import SparkSession

spark = (
    # SparkSession
    # .builder
    build_data_provenance_session()
    .appName("data-provenance-notebook")
    .getOrCreate()
)

26/04/21 10:29:20 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/21 10:29:21 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


### Test context manager

In [4]:
print(spark.conf.get("spark.provenance.enabled", "false"))

with data_provenance_enabled(spark):
    print(spark.conf.get("spark.provenance.enabled", "false"))

print(spark.conf.get("spark.provenance.enabled", "false"))

false
true
false


### Create toy dataframe

In [5]:
from datetime import date

df = spark.createDataFrame([
    ("A", date(2026, 1, 15), 10.0, 90),
    ("A", date(2026, 1, 16), 10.0, 120),
    ("A", date(2026, 1, 17), 5.0, 300),
    ("B", date(2026, 1, 15), 100.0, 20),
    ("B", date(2026, 1, 16), 100.0, 30),
    ("B", date(2026, 1, 17), 80.0, 60),
], ["product", "date", "price", "sales"]
)
df.printSchema()

df.toPandas()

root
 |-- product: string (nullable = true)
 |-- date: date (nullable = true)
 |-- price: double (nullable = true)
 |-- sales: long (nullable = true)



,product,date,price,sales
0,A,2026-01-15,10.0,90
1,A,2026-01-16,10.0,120
2,A,2026-01-17,5.0,300
3,B,2026-01-15,100.0,20
4,B,2026-01-16,100.0,30
5,B,2026-01-17,80.0,60


### Test with pyspark syntax

In [6]:
df2 = df.select("product")

df2.toPandas()

,product
0,A
1,A
2,A
3,B
4,B
5,B


In [7]:
with data_provenance_enabled(spark):
    df3 = df.select("product")

df3.toPandas()

,product,_provenance_tag
0,A,8589934592
1,A,25769803776
2,A,34359738368
3,B,51539607552
4,B,68719476736
5,B,77309411328


### Test with SQL syntax

In [8]:
df.createOrReplaceTempView("sales")

spark.sql("select * from sales").toPandas()

,product,date,price,sales
0,A,2026-01-15,10.0,90
1,A,2026-01-16,10.0,120
2,A,2026-01-17,5.0,300
3,B,2026-01-15,100.0,20
4,B,2026-01-16,100.0,30
5,B,2026-01-17,80.0,60


In [9]:
df.createOrReplaceTempView("sales")

with data_provenance_enabled(spark):
    res = spark.sql("select * from sales").toPandas()

res

,product,date,price,sales,_provenance_tag
0,A,2026-01-15,10.0,90,8589934592
1,A,2026-01-16,10.0,120,25769803776
2,A,2026-01-17,5.0,300,34359738368
3,B,2026-01-15,100.0,20,51539607552
4,B,2026-01-16,100.0,30,68719476736
5,B,2026-01-17,80.0,60,77309411328
